# Week 2, Day 1 — Impurity vs. permutation importance

Day 10 already showed, on one fixed XGBoost model, that gain-based importance and mean-|SHAP| contribution can rank the same features differently. Today pushes on a related but sharper question: two *different* scoring philosophies — impurity-based importance (read straight off the trees, free with every fit) and permutation importance (computed by breaking one feature at a time and watching test accuracy suffer) — compared across three separately-tuned models on one shared feature pipeline. The headline reason they can disagree: correlated features. When two columns carry overlapping information, impurity splits credit between them somewhat arbitrarily (whichever one a greedy split happens to grab), while permutation asks a sharper question — does the model's real test-time accuracy actually depend on this feature, given everything else it still has access to? `pclass` and `fare` are the obvious candidate pair on this dataset (lower class number, higher fare — same socioeconomic signal, two columns), and Step 2 checks that directly before the importance comparison.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, confusion_matrix
from xgboost import XGBClassifier

## Setup — one shared pipeline for all three models

Day 7 and Day 8's tree and forest were built on the `who`/`pclass`/`fare`/`family_size` feature set (manual `.map()` encoding). That set can't produce the side-by-side importance table this notebook needs, because it folds `sex` and `age` together into one categorical (`who`) and drops `embarked` entirely. So this notebook retunes the tree and forest from scratch on Day 10's richer pipeline instead — `sex`/`pclass`/`embarked` one-hot encoded via `ColumnTransformer` + `OneHotEncoder(drop="if_binary")`, `age`/`fare`/`sibsp`/`parch` numeric — so all three models see exactly the same columns and a fair comparison is actually possible.

One deliberate difference from Day 10: `age`'s missing values are median-imputed here (`pre_imputed`), not passed through raw (`pre_native`). Plain `DecisionTreeClassifier` and `RandomForestClassifier` don't share XGBoost's native missing-value handling, and today's subject is importance methods, not missingness — keeping all three models on identical, complete-data columns matters more here than it did on Day 10.

In [ ]:
df = sns.load_dataset("titanic")
df = df[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]]

x_train, x_test, y_train, y_test = train_test_split(
    df.drop("survived", axis=1), df["survived"], test_size=0.2, random_state=42
)

numeric_features = ["age", "fare", "sibsp", "parch"]
categorical_features = ["sex", "pclass", "embarked"]

pre_imputed = ColumnTransformer(
    [
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", OneHotEncoder(drop="if_binary"), categorical_features),
    ]
)

print("train shape:", x_train.shape, " test shape:", x_test.shape)

## Step 1 — retune Tree, Forest and XGBoost on the shared pipeline

Same nested-loop-plus-`cross_val_score` discipline as every prior grid search this course — no `GridSearchCV`. Tree reuses Day 7's `max_depth` grid, forest reuses Day 8's `n_estimators`×`max_depth` grid, XGBoost reuses Day 10's `n_estimators`×`max_depth`×`learning_rate` grid — same ranges, new pipeline underneath.

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, 15, None]
tree_results = []
for depth in depths:
    pipe = Pipeline(
        [
            ("pre", pre_imputed),
            ("clf", DecisionTreeClassifier(max_depth=depth, random_state=42)),
        ]
    )
    scores = cross_val_score(pipe, x_train, y_train, cv=5)
    tree_results.append((depth, scores.mean(), scores.std()))
    label = str(depth) if depth is not None else "None"
    print(f"max_depth={label:<4}: CV={scores.mean():.4f} (+/-{scores.std():.4f})")

best_depth = max(tree_results, key=lambda r: r[1])[0]
print(f"\nbest by CV mean: max_depth={best_depth}")

best_tree = Pipeline(
    [
        ("pre", pre_imputed),
        ("clf", DecisionTreeClassifier(max_depth=best_depth, random_state=42)),
    ]
)
best_tree.fit(x_train, y_train)
tree_test_preds = best_tree.predict(x_test)
tree_test_acc = accuracy_score(y_test, tree_test_preds)
print(f"\ntree test accuracy: {tree_test_acc:.4f}")
print("confusion matrix:\n", confusion_matrix(y_test, tree_test_preds))

In [ ]:
n_estimators_grid = [10, 50, 100, 200]
max_depth_grid = [3, 4, 5, 6, 8, None]
forest_results = []
for n_est in n_estimators_grid:
    for depth in max_depth_grid:
        pipe = Pipeline(
            [
                ("pre", pre_imputed),
                (
                    "clf",
                    RandomForestClassifier(
                        n_estimators=n_est, max_depth=depth, random_state=42
                    ),
                ),
            ]
        )
        scores = cross_val_score(pipe, x_train, y_train, cv=5)
        forest_results.append((n_est, depth, scores.mean(), scores.std()))

forest_results.sort(key=lambda r: -r[2])
print("top 5 forest configs by CV mean:")
for n_est, depth, mean, std in forest_results[:5]:
    print(
        f"  n_estimators={n_est:>3}, max_depth={str(depth):<4}: CV acc={mean:.4f} (+/-{std:.4f})"
    )

best_n, best_depth_f, best_cv_f, best_std_f = forest_results[0]
print(
    f"\nbest forest config: n_estimators={best_n}, max_depth={best_depth_f}, CV acc={best_cv_f:.4f} (+/-{best_std_f:.4f})"
)

best_forest = Pipeline(
    [
        ("pre", pre_imputed),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=best_n, max_depth=best_depth_f, random_state=42
            ),
        ),
    ]
)
best_forest.fit(x_train, y_train)
forest_test_preds = best_forest.predict(x_test)
forest_test_acc = accuracy_score(y_test, forest_test_preds)
print(f"\nforest test accuracy: {forest_test_acc:.4f}")
print("confusion matrix:\n", confusion_matrix(y_test, forest_test_preds))

In [ ]:
n_estimators_grid_x = [50, 100, 200]
max_depth_grid_x = [2, 3, 4]
lr_grid = [0.05, 0.1, 0.2]
xgb_results = []
for n_est in n_estimators_grid_x:
    for depth in max_depth_grid_x:
        for lr in lr_grid:
            pipe = Pipeline(
                [
                    ("pre", pre_imputed),
                    (
                        "clf",
                        XGBClassifier(
                            n_estimators=n_est,
                            max_depth=depth,
                            learning_rate=lr,
                            random_state=42,
                            eval_metric="logloss",
                        ),
                    ),
                ]
            )
            scores = cross_val_score(pipe, x_train, y_train, cv=5)
            xgb_results.append((n_est, depth, lr, scores.mean(), scores.std()))

xgb_results.sort(key=lambda r: -r[3])
print("top 5 xgb configs by CV mean:")
for n_est, depth, lr, mean, std in xgb_results[:5]:
    print(
        f"  n_estimators={n_est:>3}, max_depth={depth}, lr={lr:<4}: CV acc={mean:.4f} (+/-{std:.4f})"
    )

best_n_x, best_depth_x, best_lr_x, best_cv_x, best_std_x = xgb_results[0]
print(
    f"\nbest xgb config: n_estimators={best_n_x}, max_depth={best_depth_x}, lr={best_lr_x}, CV acc={best_cv_x:.4f} (+/-{best_std_x:.4f})"
)

best_xgb = Pipeline(
    [
        ("pre", pre_imputed),
        (
            "clf",
            XGBClassifier(
                n_estimators=best_n_x,
                max_depth=best_depth_x,
                learning_rate=best_lr_x,
                random_state=42,
                eval_metric="logloss",
            ),
        ),
    ]
)
best_xgb.fit(x_train, y_train)
xgb_test_preds = best_xgb.predict(x_test)
xgb_test_acc = accuracy_score(y_test, xgb_test_preds)
print(f"\nxgb test accuracy: {xgb_test_acc:.4f}")
print("confusion matrix:\n", confusion_matrix(y_test, xgb_test_preds))

Real numbers against the Week-1 baselines on this same 179-row test set: tree 0.7989 here vs. Day 7's 0.8212 (this pipeline's tree does *worse* — `who` folding sex and age together was a genuinely stronger single feature than sex and age split apart, for a shallow tree with a small depth budget). Forest: 0.8156 here vs. Day 8's 0.8212, a small step down. XGBoost: 0.8212 here vs. Day 10's 0.8156 — the one model that improved, once age's missingness is imputed instead of left native. None of this is the point of today's notebook, but it's worth naming rather than skating past: changing the feature representation changes accuracy too, not just importance rankings — the two aren't independent.

In [ ]:
tree_names = list(best_tree.named_steps["pre"].get_feature_names_out())
forest_names = list(best_forest.named_steps["pre"].get_feature_names_out())
xgb_names = list(best_xgb.named_steps["pre"].get_feature_names_out())
assert tree_names == forest_names == xgb_names
print("shared feature columns:", tree_names)

## Step 2 — checking the multicollinearity claim, not assuming it

`pclass` and `fare` are the obvious redundant pair here — cabin class and ticket price are two measurements of roughly the same thing (socioeconomic status). Before leaning on that in Step 5, check it directly rather than asserting it: a simple correlation between `pclass` (1/2/3) and `fare`, plus mean fare by class.

In [ ]:
corr = df[["pclass", "fare"]].corr().iloc[0, 1]
print(f"corr(pclass, fare) = {corr:.4f}")
print()
print(df.groupby("pclass")["fare"].mean())

`corr(pclass, fare) = -0.5495` — moderately strong, and in the expected direction (lower class *number* = higher fare, since 1st class is the priciest). Mean fare by class makes the redundancy concrete: 1st class averages 84.15, 2nd 20.66, 3rd 13.68 — a passenger's fare is already telling the model most of what `pclass` would. That's the setup for Step 5: if a model can reconstruct most of `pclass`'s signal from `fare` alone (or vice versa), permuting one of them shouldn't hurt much, even if it earned real credit at split time.

## Step 3 — impurity importance, side by side

`.named_steps["clf"].feature_importances_` from each of the three tuned pipelines, collapsed from one-hot columns back to their parent feature (`cat__pclass_1` + `cat__pclass_2` + `cat__pclass_3` → `pclass`, same for `sex` and `embarked`) so the three models' importances line up on the same seven rows.

In [ ]:
def parent_feature(name):
    name = name.split("__", 1)[1]
    for base in ["pclass", "sex", "embarked"]:
        if name.startswith(base + "_"):
            return base
    return name


models = {"tree": best_tree, "forest": best_forest, "xgboost": best_xgb}
imp_table = {}
for label, model in models.items():
    names = model.named_steps["pre"].get_feature_names_out()
    imps = model.named_steps["clf"].feature_importances_
    agg = {}
    for n, v in zip(names, imps):
        p = parent_feature(n)
        agg[p] = agg.get(p, 0) + v
    imp_table[label] = agg

parents = sorted(set(p for t in imp_table.values() for p in t))
print(f"{'feature':<12}" + "".join(f"{m:>10}" for m in models))
for p in parents:
    row = "".join(f"{imp_table[m].get(p, 0):>10.4f}" for m in models)
    print(f"{p:<12}{row}")

`sex` and `pclass` lead all three models — the same "women and children first, and class mattered" signal Day 1-2's EDA, Day 6's logistic regression, and Day 7's tree already converged on independently. The one number worth flagging before Step 4: the random forest gives `fare` **0.2091** — second place, ahead of `pclass`'s 0.1259, and not far behind `sex`. Tree and XGBoost don't rate `fare` nearly that highly (0.0612 and 0.0400). That's the exact kind of single-model outlier Step 2's correlation finding predicts trouble for — watch what happens to the forest's `fare` number specifically once permutation gets a turn.

## Step 4 — permutation importance, on the test set only

```python
from sklearn.inspection import permutation_importance
perm = permutation_importance(model, x_test, y_test, n_repeats=30, random_state=42, scoring="accuracy")
```

Two choices worth explaining, not just making silently:

**Test set, not train.** The question permutation importance answers is "how much does the model's *real generalization performance* depend on this feature" — that's a test-set question, the mirror image of every CV-on-train-only discipline this course has followed since Day 7 (there, train-only CV picks hyperparameters honestly; here, test-only permutation scores honestly).

**Permuting the raw column, before encoding — not the one-hot dummy after encoding.** `permutation_importance` is called on the *whole pipeline* (`pre_imputed` + classifier) with the raw `x_test`, so it shuffles `pclass`'s three raw values (1/2/3) together as one column, then re-encodes. Shuffling a single dummy column after encoding (e.g. `cat__pclass_3` alone, independent of `cat__pclass_1`/`cat__pclass_2`) would create rows with two class dummies lit at once, or none at all — combinations that never occur in real data and that the model was never trained to handle. That's not a hypothetical concern: doing it that way on this model shows `cat__pclass_3` alone scoring +0.0842 for XGBoost, nearly as high as `pclass`'s honest combined permutation score of 0.1048 — a number that looks meaningful but is partly an artifact of feeding the model nonsense rows, not a clean read of `pclass`'s real importance.

In [ ]:
raw_columns = list(x_test.columns)
print("raw columns:", raw_columns)
print()

for label, model in models.items():
    perm = permutation_importance(
        model, x_test, y_test, n_repeats=30, random_state=42, scoring="accuracy"
    )
    order = np.argsort(-perm.importances_mean)
    print(f"=== {label} ===")
    for i in order:
        print(
            f"  {raw_columns[i]:<10}: {perm.importances_mean[i]:+.4f} (+/-{perm.importances_std[i]:.4f})"
        )
    print()

## Step 5 — the reveal, in this notebook's own numbers

The clean version of the effect shows up in exactly the model Step 3 flagged: the **random forest**. `fare` goes from **0.2091** (2nd place, ahead of `pclass`) in impurity importance to **0.0136** (5th place, barely above the two weakest features) in permutation importance — a 15x drop, and a real reordering, not noise (`fare`'s permutation std is 0.0129, comfortably smaller than the 0.042 point-gap it needs to close to retake 2nd place from `age`). `pclass`, meanwhile, holds essentially the same standing in both measures for the forest (4th by impurity, 2nd by permutation — actually *moves up*).

The mechanism is exactly Step 2's finding: `fare` and `pclass` carry overlapping socioeconomic signal (`corr=-0.5495`). The forest's greedy splits found `fare` a genuinely useful column to split on early and often — hence the high impurity credit — but once `fare` is shuffled away, the forest still has `pclass` sitting right there carrying most of the same information, so test accuracy barely moves. Impurity importance measures "how much did this feature get used," which doesn't distinguish "used because irreplaceable" from "used because it happened to be available and correlated with something irreplaceable." Permutation importance measures the second thing directly, by actually taking the feature away and checking.

Tree and XGBoost show a smaller version of the same story, worth reporting honestly rather than inflating: the tree's `fare` importance was already modest by impurity (0.0612, 4th place, behind `age`) and stays at 4th under permutation (0.0160) — no reordering at all for the tree, because a depth-3 tree has few splits to spend and never leaned on `fare` heavily enough for permutation to have anything dramatic to reveal. XGBoost's `fare` actually holds its relative rank (5th impurity, 4th permutation) — its splits already favored `pclass` (0.3228 impurity, the highest of any model) over `fare`, so there was less redundant credit sitting on `fare` to begin with. The lesson isn't "fare is unimportant" or "pclass always wins" — it's that *which* correlated feature ends up over-credited by impurity depends on which one the model's specific greedy splits happened to grab first, and only permutation importance, run on held-out data, catches it after the fact.

## Step 6 — a fourth angle: L2 logistic regression coefficients

One more independent read on the same question, from a model family with no impurity concept at all. `LogisticRegression`'s default penalty is L2 (Day 6 already built this from scratch and derived why); fit here on the identical `pre_imputed` pipeline so the coefficients are on the same one-hot columns as everything above.

In [ ]:
logreg = Pipeline([("pre", pre_imputed), ("clf", LogisticRegression(max_iter=1000))])
logreg.fit(x_train, y_train)
logreg_test_preds = logreg.predict(x_test)
logreg_test_acc = accuracy_score(y_test, logreg_test_preds)

names = logreg.named_steps["pre"].get_feature_names_out()
coefs = logreg.named_steps["clf"].coef_[0]

print(f"logistic regression test accuracy: {logreg_test_acc:.4f}\n")
for n, c in sorted(zip(names, coefs), key=lambda t: -abs(t[1])):
    print(f"  {n:<20}: {c:+.4f}")

### A scale caveat, before reading these as "importances"

`fare` ranges from 0 to 512 in the training data (std ≈ 52); `sex_male` only ever takes the value 0 or 1. A logistic regression coefficient is "change in log-odds per one-unit increase" — for `fare` that's the effect of one extra *dollar*, for `sex_male` that's the effect of the entire category swing. Comparing those two raw coefficients directly, like the printout above does, isn't a fair comparison — `fare`'s coefficient looks tiny partly because one dollar is a tiny fraction of its actual range, not only because `fare` carries little information.

Fix: rescale each numeric coefficient by that feature's own standard deviation, so every number reads as "effect of one typical (1-std) swing in this feature" — directly comparable to a dummy's coefficient, which is already the effect of its one possible swing (0 to 1).

In [ ]:
numeric_cols = ["age", "fare", "sibsp", "parch"]
numeric_stds = x_train[numeric_cols].fillna(x_train[numeric_cols].median()).std()

scaled = []
for n, c in zip(names, coefs):
    base = n.split("__", 1)[1]
    effect = c * numeric_stds[base] if base in numeric_stds.index else c
    scaled.append((n, c, effect, base in numeric_stds.index))

print("per-typical-swing effect (dummy coefficients are already a full 0->1 swing):\n")
for n, c, effect, is_numeric in sorted(scaled, key=lambda t: -abs(t[2])):
    tag = f"x std={numeric_stds[n.split('__', 1)[1]]:.2f}" if is_numeric else "(dummy)"
    print(f"  {n:<20}: raw={c:+.4f}  {tag:<14} -> per-typical-swing={effect:+.4f}")

Coefficient magnitude broadly echoes permutation importance's story, once compared fairly. `sex_male` (-2.5763) and the `pclass` dummies (`pclass_3` -0.9666, `pclass_1` +0.7286) carry by far the largest per-swing effects — consistent with `sex` and `pclass` dominating both impurity and permutation importance across all three tree ensembles. Rescaled to a per-typical-swing basis (previous cell), `fare`'s effect is **+0.19** — 9th of 12, clearly behind `sex`, all three `pclass` dummies, `age`, and `sibsp`. Not literally the smallest coefficient in the model, though (that's `embarked_Q` at -0.0013, then `parch` at -0.10) — the raw, unscaled coefficient (+0.0037) looked far smaller than that only because `fare`'s dollar-valued scale makes a "one-unit" swing tiny relative to its actual range. Corrected for scale, `fare`'s effect is modest but not negligible — still a fourth, structurally independent read that lines up with Step 5 (once `pclass` is already in the model, `fare` has comparatively little *additional* linear signal to contribute), just less dramatically "near zero" than the raw coefficient alone suggested.

The through-line for today, across four different scoring methods on three different model families: correlated features don't have one "true" importance that every method agrees on. Impurity importance and raw coefficient magnitude can both mislead — impurity by crediting whichever correlated feature the fitting process happened to lean on, raw coefficients by conflating a feature's information content with its measurement scale. Permutation importance is the one method here that sidesteps both: shuffle the raw feature, keep everything else fixed, and check whether the model's real accuracy actually notices. On this dataset, for the random forest specifically, `fare` is the feature that answer turns out to be "no" for.

# Week 2, Day 2 — Threshold tuning: F1-optimal, Youden's J, and cost-sensitive selection

Every model above was judged by `.predict()` — sklearn's default, which silently converts `predict_proba() >= 0.5` into a class label. That cutoff was never chosen; it was just inherited. Today asks whether 0.5 is actually the right place to draw the line, using `best_forest` from Day 1 as-is — no retraining, same model, just moving where the probability gets rounded to 0 or 1.

Three escalating answers: sweep every possible threshold and optimize a metric that balances precision and recall (F1), then one that balances the two classes symmetrically (Youden's J), then — the one that actually matters in a real deployment — a threshold chosen to minimize the *real cost* of the two error types, once someone states what a false positive and a false negative actually cost.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve, f1_score

probs = best_forest.predict_proba(x_test)[:, 1]

preds_default = (probs >= 0.5).astype(int)
f1_default = f1_score(y_test, preds_default)
cm_default = confusion_matrix(y_test, preds_default)

print(f"threshold=0.5   F1={f1_default:.4f}")
print("confusion matrix:\n", cm_default)
print(
    "\nmatches Day 1's forest confusion matrix exactly:",
    (cm_default == confusion_matrix(y_test, forest_test_preds)).all(),
)

`[[94, 11], [22, 52]]` — identical to Day 1's forest confusion matrix, confirming `.predict()` is nothing more than `predict_proba() >= 0.5` under the hood. F1 at that cutoff: **0.7591**. Everything below moves the cutoff and asks whether 0.7591 is actually the best this same model can do.

## Block 2 — Threshold sweep: F1-optimal and Youden's J

A ROC curve and a precision-recall curve are both, literally, the result of sweeping the decision threshold from 1 down to 0 and recomputing precision/recall (or TPR/FPR) at every cutoff — `precision_recall_curve` and `roc_curve` just return the whole sweep at once instead of one point. Since F1 is a function of precision and recall, computing F1 across that whole sweep and taking the max finds the threshold that's F1-optimal, with no retraining and no grid search over the model itself — only over where its output gets cut.

One gotcha worth calling out rather than silently coding around: `precision_recall_curve` returns one more `(precision, recall)` pair than it does thresholds — the last point is the threshold-less corner (recall=0, precision=1, reached only as the threshold approaches infinity). So `f1_scores` has to be sliced to drop that last, threshold-less entry before taking `argmax` and indexing back into `thresh`, or the index can point past the end of the thresholds array.

In [ ]:
prec, rec, thresh = precision_recall_curve(y_test, probs)
f1_scores = 2 * prec * rec / (prec + rec + 1e-12)

best_idx = np.argmax(f1_scores[:-1])  # drop the threshold-less last point
best_thresh_f1 = thresh[best_idx]

print(f"default   threshold=0.500  F1={f1_default:.4f}")
print(
    f"F1-optimal threshold={best_thresh_f1:.3f}  F1={f1_scores[best_idx]:.4f}  "
    f"precision={prec[best_idx]:.4f}  recall={rec[best_idx]:.4f}"
)

Moving the cutoff from 0.5 to **0.318** — same model, no retraining — lifts F1 from 0.7591 to **0.8148**: precision drops from 0.8254 (52/(52+11)) to 0.75, recall climbs from 0.7027 (52/(52+22)) to 0.892. Lowering the threshold means the forest only needs to be ~32% confident to call someone a survivor, so it now catches more of the true survivors it was previously too cautious to flag — at the cost of a few more false alarms. That's the entire trade a threshold controls: it doesn't change what the model *knows*, only how willing it is to act on partial confidence.

In [ ]:
fpr, tpr, roc_thresh = roc_curve(y_test, probs)
j_scores = tpr - fpr  # Youden's J

best_j_idx = np.argmax(j_scores)
best_thresh_j = roc_thresh[best_j_idx]

print(
    f"Youden's J-optimal threshold={best_thresh_j:.3f}  J={j_scores[best_j_idx]:.4f}  "
    f"(TPR={tpr[best_j_idx]:.4f}, FPR={fpr[best_j_idx]:.4f})"
)

**0.318 again** — the same threshold F1 picked. Worth flagging as a coincidence of this particular model and dataset, not a rule: F1 balances precision and recall on the positive class specifically (it never looks at true negatives at all), while Youden's J balances the true-positive rate against the false-positive rate symmetrically across *both* classes. They're answering genuinely different questions — F1 asks "how good is this model at finding survivors without crying wolf," J asks "how well does this model separate the two classes overall" — and on an imbalanced or asymmetric-cost problem they can easily land on different cutoffs. Here they happen to agree because `best_forest`'s probability distribution for the two classes is well-separated enough that both balancing criteria peak in the same place.

The numbers above say *where* the optimal threshold sits, not what the tradeoff looks like around it — plotting both curves, with the default and each optimal point marked, shows that directly. PR is the more honest of the two here: `survived` isn't wildly imbalanced (300/549 ≈ 40% positive), but ROC's false-positive rate is measured against all negatives, so it can still look deceptively strong even where precision is mediocre — PR doesn't have that cushion.

In [ ]:
import matplotlib.pyplot as plt

default_pr_idx = np.argmin(np.abs(thresh - 0.5))
default_roc_idx = np.argmin(np.abs(roc_thresh - 0.5))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(rec, prec, color="steelblue")
axes[0].scatter(
    rec[best_idx],
    prec[best_idx],
    color="crimson",
    zorder=5,
    label=f"F1-optimal (t={best_thresh_f1:.2f})",
)
axes[0].scatter(
    rec[default_pr_idx],
    prec[default_pr_idx],
    color="black",
    marker="x",
    zorder=5,
    label="default (t=0.5)",
)
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall curve")
axes[0].legend()

axes[1].plot(fpr, tpr, color="darkorange")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
axes[1].scatter(
    fpr[best_j_idx],
    tpr[best_j_idx],
    color="crimson",
    zorder=5,
    label=f"Youden's J-optimal (t={best_thresh_j:.2f})",
)
axes[1].scatter(
    fpr[default_roc_idx],
    tpr[default_roc_idx],
    color="black",
    marker="x",
    zorder=5,
    label="default (t=0.5)",
)
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC curve")
axes[1].legend()

plt.tight_layout()
plt.show()

## Block 3 — Cost-sensitive selection: the actual payoff

Neither F1 nor Youden's J knows anything about this problem's real stakes — both treat a false positive and a false negative as interchangeable, just in different ratios baked into the formula. In a real deployment they almost never are: for a survival predictor used to prioritize rescue effort, a false negative (predicting death for someone who actually survives — an at-risk person never flagged) is plausibly far worse than a false positive (flagging someone who was never in danger). "Missing someone who needed help is worse than a wasted check" isn't just an intuition — it's a cost ratio, and every cost ratio has a specific, computable optimal threshold attached to it. The fix: assign real costs to each error type and let the threshold fall out of minimizing total cost, instead of eyeballing 0.5 or optimizing a metric that never asked what the errors actually cost.

In [ ]:
def total_cost(threshold, probs, y_true, cost_fp, cost_fn):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return fp * cost_fp + fn * cost_fn


candidate_thresholds = np.linspace(0.01, 0.99, 99)

for cost_fp, cost_fn in [(1, 1), (1, 5), (1, 10)]:
    costs = [
        total_cost(t, probs, y_test, cost_fp, cost_fn) for t in candidate_thresholds
    ]
    best_i = np.argmin(costs)
    best_t, best_c = candidate_thresholds[best_i], costs[best_i]

    preds_best = (probs >= best_t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds_best).ravel()
    default_c = total_cost(0.5, probs, y_test, cost_fp, cost_fn)

    print(
        f"cost_fp={cost_fp:>2}, cost_fn={cost_fn:>2}:  "
        f"optimal threshold={best_t:.2f}, cost={best_c:.0f} (FP={fp}, FN={fn})  |  "
        f"cost at default 0.5={default_c:.0f}"
    )

Under **equal cost**, the threshold barely matters: 0.51 (cost=30, FP=8/FN=22) is only marginally better than the default 0.5 (cost=33, FP=11/FN=22) — when both error types are worth the same, F1's balance point and the cost-optimal point are close by construction. But as false negatives get genuinely more expensive, the gap explodes. At **5x** (missing a survivor costs 5x a false alarm), the properly-chosen threshold (0.24) drives cost down to 58 versus 121 at the default — under half, achieved by accepting more false positives (28 vs. 11) to cut false negatives from 22 to 6. At **10x**, the optimal threshold crashes to 0.14, accepting 50 false alarms specifically to push false negatives down to 3 — cost 80 versus 231 at the default, less than a third.

This is the concrete version of the intuition from the top of this block. "Missing someone who needed help is worse than a wasted check" isn't just a feeling anymore — it's a cost ratio, and that ratio has a specific, computable optimal threshold attached to it. Whoever deploys a classifier for a decision with asymmetric stakes needs to have this conversation explicitly — what does a missed case really cost, versus an extra false alarm — rather than silently inheriting 0.5 from `.predict()`.

# Week 2, Day 3 — Probability calibration: Brier score, reliability diagrams, and when (not) to fix it

Day 2's cost-sensitive threshold selection treated `predict_proba()`'s output as real probabilities — plugged straight into a cost formula and swept for the minimum. Today checks whether that trust was actually earned. A model can be highly accurate while being badly **calibrated**: a well-calibrated model's "70% confident" predictions should be right about 70% of the time, and accuracy alone can't detect a violation of that, because accuracy only ever looks at the argmax label, never at whether the stated confidence behind it matches reality.

## Block 2 — Check all three models

`brier_score_loss` is the mean squared error between predicted probability and the actual 0/1 outcome — lower is better, 0 is perfect, and unlike accuracy it's sensitive to *how* confident a wrong (or right) call was, not just whether it crossed 0.5. `calibration_curve` buckets predictions into bins by predicted probability and reports, per bin, what fraction of those rows were actually positive — the gap between "predicted" and "actual" in each bin is calibration error made visible.

In [ ]:
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

probs_logreg = logreg.predict_proba(x_test)[:, 1]
probs_xgb = best_xgb.predict_proba(x_test)[:, 1]
# probs (forest) already computed in Day 2

for label, p in [("logistic regression", probs_logreg), ("random forest", probs), ("xgboost", probs_xgb)]:
    print(f"{label:<20}: Brier={brier_score_loss(y_test, p):.4f}")

Random forest is best-calibrated (**0.1257**), XGBoost close behind (**0.1274**), logistic regression worst (**0.1342**) — despite being the simplest model of the three. Calibration quality doesn't track model complexity or even accuracy ranking (XGBoost has the highest test accuracy, 0.8212, but isn't the best-calibrated). Worth a closer look at XGBoost specifically, since it's the tree ensemble with the least natural averaging (a single boosted sequence vs. the forest's 100-tree bagged average, which tends to smooth probabilities toward the middle almost for free).

In [ ]:
frac_pos, mean_pred = calibration_curve(y_test, probs_xgb, n_bins=5, strategy="quantile")

print("XGBoost reliability diagram (5 quantile bins):\n")
for mp, fp in zip(mean_pred, frac_pos):
    print(f"predicted~{mp:.3f}  actual_freq={fp:.3f}  gap={fp - mp:+.3f}")

Gaps are modest — the largest is **-0.097** (predicted ~0.125, actual 0.028), the rest under 0.06 — nothing on the scale of a dramatic, structural miscalibration. Worth being honest about why some of this is noise rather than signal: with 179 test rows split into 5 quantile bins, each bin holds only ~36 rows, so a single flipped outcome shifts that bin's observed frequency by about 2.8 percentage points. The −0.097 gap in the second bin could be one or two unlucky rows, not a real pattern the model is consistently wrong about.

The general principle these tools exist for still holds regardless of how dramatic this particular case is: a model's *confidence* can keep sharpening — moving Brier score, moving log-loss — well after its *accuracy* has already flatlined, because accuracy only ever looks at whether the argmax crossed the threshold, never at how far the probability actually sits from the truth. Brier score and reliability diagrams are the tools that catch that; accuracy alone cannot.

## Block 3 — Try fixing it on XGBoost

Even without a dramatic miscalibration to fix, it's worth walking through the tool: `CalibratedClassifierCV` re-maps raw scores to better-calibrated probabilities two ways — `method="sigmoid"` (Platt scaling) fits a 1-D logistic curve through the score; `method="isotonic"` fits an unconstrained monotonic step function, more flexible, more data-hungry.

One mechanical detail worth stating up front rather than discovering by surprise: with `cv=5`, `CalibratedClassifierCV` doesn't calibrate `best_xgb`'s own fitted score directly — it retrains 5 clones of the model on 5 different folds of `x_train`, calibrates each clone on its held-out fold, and averages the five resulting probabilities. That average is not a strict monotonic function of `best_xgb`'s own original score, so — contrary to what "it's just a monotonic rescaling" might suggest — individual predictions genuinely can flip under either method, sigmoid included. Diffing the actual prediction arrays below, not just comparing accuracy totals, is what catches that.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV


def calibration_report(label, base_model, x_train, y_train, x_test, y_test):
    preds_before = base_model.predict(x_test)
    probs_before = base_model.predict_proba(x_test)[:, 1]
    print(f"{label}")
    print(f"  before calibration: Brier={brier_score_loss(y_test, probs_before):.4f}  acc={accuracy_score(y_test, preds_before):.4f}")

    for method in ["sigmoid", "isotonic"]:
        cal = CalibratedClassifierCV(base_model, method=method, cv=5)
        cal.fit(x_train, y_train)
        preds_after = cal.predict(x_test)
        probs_after = cal.predict_proba(x_test)[:, 1]

        flipped = preds_before != preds_after
        fixed = ((preds_before != y_test.values) & (preds_after == y_test.values) & flipped).sum()
        broken = ((preds_before == y_test.values) & (preds_after != y_test.values) & flipped).sum()

        print(f"  after {method:<8}: Brier={brier_score_loss(y_test, probs_after):.4f}  acc={accuracy_score(y_test, preds_after):.4f}  "
              f"(flipped={flipped.sum()}: fixed={fixed}, broken={broken})")


calibration_report("xgboost", best_xgb, x_train, y_train, x_test, y_test)

Both methods make Brier score *worse*, not better — the honest outcome, not the "it obviously helps" story. With 712 training rows split 5 ways inside `CalibratedClassifierCV`, each fold trains its recalibration map on only ~142 rows, and XGBoost's calibration wasn't broken badly enough here to be worth the variance that adds. Isotonic is worse than sigmoid specifically because its extra flexibility gives it more capacity to fit fold-specific noise instead of a real trend.

On accuracy: sigmoid's flip count (2) splits evenly — 1 prediction fixed, 1 broken — so accuracy comes out looking "unchanged" (0.8212 both), but that's real churn canceling out, not proof that sigmoid left every decision alone. Isotonic's flips (also 2) both happened to break a previously-correct call, so its accuracy genuinely drops (0.8212 → 0.8101). Same flip count, opposite direction, because which specific rows sit near the decision boundary — not the method's abstract properties — decided the outcome.

## Block 4 — the same fix, on the model that needed it least

Apply the identical correction to `best_forest` — already the best-calibrated model of the three (Brier 0.1257).

In [ ]:
calibration_report("random forest", best_forest, x_train, y_train, x_test, y_test)

Same direction as XGBoost — both methods make Brier worse, isotonic worse than sigmoid — and accuracy lands "unchanged" for both methods here too. But per Block 3's corrected mechanism, that's again real predictions flipping in both directions and exactly canceling (fixed and broken counts matching), not proof that nothing moved.

The actual lesson this dataset teaches: calibration correction isn't a free pass to apply everywhere, and "the more complex model must need it more" isn't a safe assumption either — random forest and XGBoost's Brier scores differ by only 0.0017, nowhere near enough to justify assuming one obviously needs fixing and the other doesn't. Check with Brier score and a reliability diagram first (Block 2), and only reach for `CalibratedClassifierCV` when that check actually shows a real gap; on a modest training set, default to sigmoid over isotonic if you do reach for it, since isotonic's extra flexibility is exactly what let it fit fold noise instead of signal here for both models.

Tying back to Day 2: the cost-sensitive threshold work assumed `predict_proba()` was trustworthy. Today's check says that assumption held up reasonably well for `best_forest` specifically — but the discipline of *verifying* it, not this particular verdict, is the actual takeaway. A different model, or a larger training set, could easily have gone the other way.